https://investors.sgx.com/market/stock-screener?from=/market/securities

---

# Setup & Import

In [46]:
# Setup
%load_ext autoreload
%autoreload 2
#%load_ext jupyter_ai_magics
import sys; sys.path.append('..')
from src.env_setup import init_analysis_env; init_analysis_env()

df_ = load_local_data("Stock Screener Data*.csv*", na_values=['-'], dtype={'Code': str})
df_.head()

# Cleaning

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
🚀 Analysis environment initialized: pd, np, yf, plt, sns, load_local_data, datetime, timedelta, xw are ready.
Loading: /home/mrsmmori/notebooks/Win_Downloads/Stock Screener Data 7 Jun 11-8-22.csv.csv


,Trading Name,Code,Last Price,ROE %,Mkt Cap ($M),Tot. Rev ($M),P/E,Yield (%),Sector,GTI Score,4-wk %Pr. Chg.,13-wk %Pr. Chg.,26-wk %Pr. Chg.,52-wk %Pr. Chg.,Net Profit,Price/CF,Debt/Equity,1-yr %Rev. Chg.,Price/Book Value
0,Tai Sin Electric,500,0.55,12.09,SGD 250.843,SGD 527.837,14.30,4.31,Industrials,78.00,-5.22,5.24,-5.56,38.66,0.05,NaN,0.04,19.98,1.12
1,HS Optimus,504,0.01,5.73,SGD 37.664,SGD 13.551,13.73,NaN,Consumer Cyclical,67.00,0.00,133.33,250.00,250.00,0.20,74.07,0.01,-6.18,0.77
2,AsiaMedic,505,0.02,13.34,SGD 26.4,SGD 35.221,13.14,NaN,Healthcare,66.00,0.00,21.05,21.05,91.67,0.03,6.85,0.96,21.81,1.64
3,Fuji Offset,508,0.73,2.33,SGD 43.736,SGD 3.274,48.34,0.68,Industrials,70.00,28.95,5.76,36.11,-10.91,0.28,357.14,0.00,-9.76,1.14
4,DISA,532,0.00,-137.72,SGD 14.09,SGD 2.667,NaN,NaN,Technology,34.00,0.00,0.00,0.00,0.00,-0.42,NaN,0.01,-19.20,3.67


In [47]:
def make_clickable(val):
    if pd.notna(val):
        return f'<a target="_blank" href="{val}">Link to SGX</a>'
    return ""

# Data Clearning

In [48]:
df = df_.copy()

In [49]:
rename_dict = {
    'Trading Name':    'name',
    'Code':            'code',
    'Last Price':      'price',
    'ROE %':           'roe',
    'Mkt Cap ($M)':    'mkt_cap',
    'Tot. Rev ($M)':   'rev',
    'P/E':             'pe',
    'Yield (%)':       'yield',
    'Sector':          'sector',
    'GTI Score':       'gti',
    '4-wk %Pr. Chg.':  'chg_4w',
    '13-wk %Pr. Chg.': 'chg_13w',
    '26-wk %Pr. Chg.': 'chg_26w',
    '52-wk %Pr. Chg.': 'chg_52w',
    'Net Profit':      'net_profit',
    'Price/CF':        'p_cf',
    'Debt/Equity':     'de',
    '1-yr %Rev. Chg.': 'rev_chg_1y',
    'Price/Book Value':'pb'
}

df = df.rename(columns=rename_dict)

In [50]:
string_cols = ["name", "code", "sector"]
for col in string_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()

def clean_numeric(series):
    if series.dtype == "object":
        series = (
            series.astype(str)
            .str.replace(",", "", regex=False)
            .str.replace("%", "", regex=False)
            .str.replace("$", "", regex=False)
            .str.replace(" ", "", regex=False)
        )
        series = pd.to_numeric(series, errors="coerce")
    return series


numeric_cols = [
    "price",
    "roe",
    "mkt_cap",
    "rev",
    "pe",
    "yield",
    "gti",
    "chg_4w",
    "chg_13w",
    "chg_26w",
    "chg_52w",
    "net_profit",
    "p_cf",
    "de",
    "rev_chg_1y",
    "pb",
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = clean_numeric(df[col])

if "code" in df.columns:
    df = df.drop_duplicates(subset=["code"], keep="first")

print(df.dtypes)

name              str
code              str
price         float64
roe               str
mkt_cap           str
rev               str
pe            float64
yield         float64
sector            str
gti           float64
chg_4w        float64
chg_13w       float64
chg_26w       float64
chg_52w           str
net_profit    float64
p_cf          float64
de            float64
rev_chg_1y        str
pb            float64
dtype: object


In [51]:
list(df["sector"].unique())

['Industrials',
 'Consumer Cyclical',
 'Healthcare',
 'Technology',
 'Basic Materials',
 'Consumer Defensive',
 'Real Estate',
 'Energy',
 'Utilities',
 'Financial Services',
 nan,
 'Communication Services']

In [52]:
df.head(5)

,name,code,price,roe,mkt_cap,rev,pe,yield,sector,gti,chg_4w,chg_13w,chg_26w,chg_52w,net_profit,p_cf,de,rev_chg_1y,pb
0,Tai Sin Electric,500,0.55,12.09,SGD 250.843,SGD 527.837,14.30,4.31,Industrials,78.00,-5.22,5.24,-5.56,38.66,0.05,NaN,0.04,19.98,1.12
1,HS Optimus,504,0.01,5.73,SGD 37.664,SGD 13.551,13.73,NaN,Consumer Cyclical,67.00,0.00,133.33,250.00,250.00,0.20,74.07,0.01,-6.18,0.77
2,AsiaMedic,505,0.02,13.34,SGD 26.4,SGD 35.221,13.14,NaN,Healthcare,66.00,0.00,21.05,21.05,91.67,0.03,6.85,0.96,21.81,1.64
3,Fuji Offset,508,0.73,2.33,SGD 43.736,SGD 3.274,48.34,0.68,Industrials,70.00,28.95,5.76,36.11,-10.91,0.28,357.14,0.00,-9.76,1.14
4,DISA,532,0.00,-137.72,SGD 14.09,SGD 2.667,NaN,NaN,Technology,34.00,0.00,0.00,0.00,0.00,-0.42,NaN,0.01,-19.20,3.67


In [53]:
def clean_currency_to_float(series):
    cleaned = (
        series.astype(str)
        .str.replace("SGD", "", regex=False)
        .str.replace(" ", "", regex=False)
        .str.replace(",", "", regex=False)
    )
    return pd.to_numeric(cleaned, errors="coerce")


# 'mkt_cap' と 'rev' を両方とも数値型（float）に変換
df["mkt_cap"] = clean_currency_to_float(df["mkt_cap"])
df["rev"] = clean_currency_to_float(df["rev"])

In [54]:
df.head(5)

,name,code,price,roe,mkt_cap,rev,pe,yield,sector,gti,chg_4w,chg_13w,chg_26w,chg_52w,net_profit,p_cf,de,rev_chg_1y,pb
0,Tai Sin Electric,500,0.55,12.09,250.84,527.84,14.30,4.31,Industrials,78.00,-5.22,5.24,-5.56,38.66,0.05,NaN,0.04,19.98,1.12
1,HS Optimus,504,0.01,5.73,37.66,13.55,13.73,NaN,Consumer Cyclical,67.00,0.00,133.33,250.00,250.00,0.20,74.07,0.01,-6.18,0.77
2,AsiaMedic,505,0.02,13.34,26.40,35.22,13.14,NaN,Healthcare,66.00,0.00,21.05,21.05,91.67,0.03,6.85,0.96,21.81,1.64
3,Fuji Offset,508,0.73,2.33,43.74,3.27,48.34,0.68,Industrials,70.00,28.95,5.76,36.11,-10.91,0.28,357.14,0.00,-9.76,1.14
4,DISA,532,0.00,-137.72,14.09,2.67,NaN,NaN,Technology,34.00,0.00,0.00,0.00,0.00,-0.42,NaN,0.01,-19.20,3.67


In [55]:
df['SGX_URL'] = np.nan
mask = df['code'].notna()
df['SGX_URL'] = df['code'].apply(
    lambda x: f'https://investors.sgx.com/market/security-details/stocks/{x}' if pd.notna(x) else None
)

In [56]:
cols = list(df.columns)
cols.remove('SGX_URL') 
cols.insert(2, 'SGX_URL') 
df = df[cols]

In [57]:
# 極限まで短くしたい場合のマッピング定義
short_mapping = {
    "Industrials": "ind",
    "Consumer Cyclical": "cyclical",
    "Healthcare": "health",
    "Technology": "tech",
    "Basic Materials": "materials",
    "Consumer Defensive": "defensive",
    "Real Estate": "estate",
    "Energy": "energy",
    "Utilities": "utilities",
    "Financial Services": "finance",
    "Communication Services": "comm",
}

sector_dfs = {}
for sector in df["sector"].unique():
    if pd.isna(sector):
        continue

    # マッピングにあればそれを使い、なければ小文字変換にする
    short_name = short_mapping.get(sector, str(sector).lower().replace(" ", "_"))

    sector_dfs[short_name] = df[df["sector"] == sector].copy()


## Industrial

In [58]:
sector_dfs['ind'].sort_values(by="rev", ascending=False, na_position="last").head(5).style.format({'SGX_URL': make_clickable})

,name,code,SGX_URL,price,roe,mkt_cap,rev,pe,yield,sector,gti,chg_4w,chg_13w,chg_26w,chg_52w,net_profit,p_cf,de,rev_chg_1y,pb
609,YZJ Shipbldg SGD,BS6,Link to SGX,3.540000,29.57,13931.986000,28504.820000,8.520000,5.650000,Industrials,75.500000,-13.430000,-11.370000,9.040000,61.90,0.290000,16.470000,nan,7.40,2.290000
287,Jardine C&C,C07,Link to SGX,28.340000,11.82,11200.996000,21358.100000,8.760000,5.110000,Industrials,107.300000,-9.370000,-13.650000,-13.570000,24.89,0.050000,2.760000,0.360000,-4.22,1.020000
481,SIA,C6L,Link to SGX,6.970000,7.19,21961.663000,20522.000000,18.150000,5.020000,Industrials,95.600000,11.340000,4.340000,10.110000,2.37,0.060000,4.230000,0.480000,5.03,1.270000
511,ST Engineering,S63,Link to SGX,10.890000,17.65,33990.949000,12346.426000,74.030000,1.650000,Industrials,105.100000,1.300000,1.850000,35.130000,39.70,0.020000,20.040000,1.240000,9.50,13.210000
259,HPH Trust SGD,P7VU,Link to SGX,0.260000,3.00,2291.382000,11610.004000,18.490000,7.250000,Industrials,nan,0.000000,4.000000,2.110000,39.42,0.060000,2.810000,0.620000,3.45,0.560000


1. 【高ROE・高配当】圧倒的な勢いと罠を併せ持つ「造船最大手」
Yangzijiang Shipbuilding (BS6 / YZJ Shipbldg)

データの特徴: ROE 29.57% という驚異的な収益性を誇り、配当利回りも 5.65% と高水準。PE（PER）は 8.52倍 と極めて割安に見えます。

背景と現状: 世界的なコンテナ船・環境対応船の需要爆発を背景に、手持ち工事残高（バックログ）が過去最高水準に達しており、純利益成長率（net_profit）も0.29（+29%）と絶好調です。

投資判断: 直近4週間（-13.43%）、13週間（-11.37%）と、株価は大きく調整を入れています。これは3月に過去最高値（4.29 SGD）をつけた後の利益確定売りの動きです。ファンダメンタルズ（企業の稼ぐ力）は崩れておらず、このセクターで「高い配当と成長」を同時に狙うなら、今回の調整局面は絶好のエントリーチャンスと言えます。

## "Consumer Cyclical": "cyclical",

In [59]:
sector_dfs['cyclical'].sort_values(by="rev", ascending=False, na_position="last").head(5).style.format({'SGX_URL': make_clickable})

,name,code,SGX_URL,price,roe,mkt_cap,rev,pe,yield,sector,gti,chg_4w,chg_13w,chg_26w,chg_52w,net_profit,p_cf,de,rev_chg_1y,pb
251,Hong Leong Asia,H22,Link to SGX,3.030000,10.72,2418.368000,5182.291000,20.090000,1.650000,Consumer Cyclical,99.300000,4.120000,6.620000,43.660000,148.39,0.020000,3.820000,0.270000,21.95,2.220000
211,Genting Sing,G13,Link to SGX,0.615000,4.73,7434.165000,2452.053000,19.040000,6.500000,Consumer Cyclical,92.200000,-8.890000,-6.620000,-13.610000,-7.75,0.170000,9.420000,0.000000,-3.08,0.910000
531,TC Auto,VI2,Link to SGX,0.022000,-507.80,12.972000,1994.988000,nan,nan,Consumer Cyclical,55.600000,-15.380000,-15.380000,-26.670000,-38.89,-0.060000,0.840000,nan,-21.90,nan
143,Combine Will,N0Z,Link to SGX,1.200000,4.84,38.793000,1810.977000,6.380000,4.170000,Consumer Cyclical,68.100000,-5.510000,-6.250000,-9.090000,16.50,0.020000,nan,0.020000,23.09,0.300000
538,TheHourGlass,AGS,Link to SGX,2.640000,17.98,1691.875000,1338.223000,9.500000,2.270000,Consumer Cyclical,62.600000,11.390000,16.300000,18.390000,60.71,0.130000,7.750000,0.170000,15.08,1.580000


## "Healthcare": "health",

In [60]:

sector_dfs['health'].sort_values(by="rev", ascending=False, na_position="last").head(10).style.format({'SGX_URL': make_clickable})

,name,code,SGX_URL,price,roe,mkt_cap,rev,pe,yield,sector,gti,chg_4w,chg_13w,chg_26w,chg_52w,net_profit,p_cf,de,rev_chg_1y,pb
267,IHH,Q0F,Link to SGX,2.830000,6.95,24920.716000,26004.000000,36.840000,1.170000,Healthcare,nan,0.350000,-1.810000,9.520000,38.31,0.070000,14.350000,0.320000,5.59,2.610000
139,CMS,8A8,Link to SGX,1.750000,8.83,4212.288000,8212.058000,15.000000,3.040000,Healthcare,nan,-19.350000,-17.040000,-20.390000,nan,0.160000,29.670000,0.000000,9.95,1.290000
333,Lonza,O6Z,Link to SGX,79.970000,-3.19,55664.397000,6531.000000,21.470000,3.750000,Healthcare,nan,10.180000,10.180000,10.180000,10.18,0.150000,14.970000,0.530000,19.18,1.400000
545,Top Glove,BVA,Link to SGX,0.255000,2.00,2083.808000,3612.412000,46.210000,0.590000,Healthcare,nan,6.250000,41.670000,35.640000,4.70,0.020000,13.890000,0.170000,38.93,1.340000
451,Riverstone,AP4,Link to SGX,0.870000,13.61,1289.486000,995.314000,19.340000,4.690000,Healthcare,74.900000,18.160000,22.340000,5.170000,32.57,0.210000,13.120000,nan,-7.23,2.720000
443,Raffles Medical,BSL,Link to SGX,0.945000,6.69,1738.980000,765.299000,24.800000,3.170000,Healthcare,81.400000,-2.500000,-4.410000,-0.510000,-1.02,0.100000,17.300000,0.030000,1.83,1.640000
539,Thomson Medical,A50,Link to SGX,0.055000,-9.80,1454.399000,408.669000,nan,nan,Healthcare,94.800000,-3.510000,-3.510000,-5.170000,27.91,-0.240000,22.940000,2.020000,12.39,3.010000
232,Haw Par,H02,Link to SGX,15.490000,6.29,3428.993000,229.967000,12.920000,2.580000,Healthcare,95.400000,-9.630000,2.950000,0.450000,33.87,1.150000,59.880000,0.010000,-6.07,0.800000
437,Q&M Dental,QC7,Link to SGX,0.585000,8.86,554.019000,197.226000,59.690000,1.400000,Healthcare,65.000000,-0.850000,13.310000,11.170000,56.11,0.070000,15.020000,1.680000,9.16,5.310000
264,Hyphens Pharma,1J5,Link to SGX,0.330000,8.29,103.471000,177.367000,17.740000,4.550000,Healthcare,90.300000,-2.820000,6.150000,-1.430000,13.11,0.050000,5.540000,0.040000,-9.24,1.450000


## "Technology": "tech",

In [61]:
sector_dfs['tech'].sort_values(by="rev", ascending=False, na_position="last").head(10).style.format({'SGX_URL': make_clickable})

,name,code,SGX_URL,price,roe,mkt_cap,rev,pe,yield,sector,gti,chg_4w,chg_13w,chg_26w,chg_52w,net_profit,p_cf,de,rev_chg_1y,pb
424,PC Partner,PCT,Link to SGX,2.240000,16.31,868.859000,13951.654000,10.770000,4.100000,Technology,nan,15.840000,90.240000,143.750000,105.26,0.040000,1.880000,0.020000,38.38,1.660000
142,Comba,STC,Link to SGX,0.220000,1.20,621.538000,4570.827000,116.840000,0.450000,Technology,nan,0.000000,-43.590000,53.850000,137.63,0.010000,7.750000,0.210000,0.94,1.310000
580,Venture,V03,Link to SGX,17.880000,7.99,5141.034000,2534.517000,22.720000,4.190000,Technology,75.900000,8.500000,18.580000,23.360000,68.90,0.090000,20.490000,0.000000,-7.36,1.840000
588,Willas-Array,BDR,Link to SGX,1.080000,11.01,111.228000,2358.253000,12.270000,nan,Technology,76.400000,-10.000000,22.730000,19.340000,227.27,0.020000,2.610000,0.000000,-6.73,1.340000
298,Karin Tech,K29,Link to SGX,0.255000,4.69,54.901000,2080.868000,nan,2.560000,Technology,66.100000,-1.920000,-5.560000,0.000000,-8.24,0.010000,2.500000,0.020000,-12.37,0.900000
578,Valuetronics,BN2,Link to SGX,1.100000,8.01,450.810000,1660.246000,23.490000,2.220000,Technology,78.400000,7.840000,30.180000,27.910000,63.43,0.090000,12.210000,nan,-3.98,1.870000
9,CSE Global,544,Link to SGX,1.380000,14.11,999.096000,968.919000,26.240000,1.880000,Technology,98.000000,1.470000,10.680000,47.580000,199.15,0.040000,nan,0.180000,12.51,3.630000
203,Frencken,E28,Link to SGX,2.920000,8.60,1251.557000,865.121000,31.910000,0.940000,Technology,72.900000,9.770000,45.200000,112.050000,154.09,0.040000,7.490000,0.120000,8.91,2.630000
470,Serial System,S69,Link to SGX,0.076000,1.76,68.768000,860.468000,23.690000,1.450000,Technology,90.400000,5.620000,15.070000,-1.150000,67.61,0.000000,nan,0.170000,9.10,0.400000
375,Multi-Chem,AWZ,Link to SGX,3.660000,17.25,329.749000,653.925000,12.470000,8.500000,Technology,84.900000,0.000000,12.210000,14.540000,28.10,0.040000,7.200000,0.000000,-4.35,2.170000


1. 【AI・グラボ特需】驚異的なリターンを叩き出す「本命・成長株」
グラフィックボード（GPU）の製造や、最先端の半導体・AI関連インフラに直結している、今まさに市場の資金を集めているグループです。
PC Partner Group (PCT)
分析: 自社ブランド（ZOTACなど）でNVIDIAなどのGPU（グラフィックボード）を製造・販売する企業です。ROE 16.31% と高く、52週間（1年間）の株価パフォーマンスは +105.26% と驚異的な大化けを演じています。
配当視点: 急騰した後でも 配当利回りは 4.10%、PE（PER）も 10.77倍 と、業績の伸びに対して依然として割安感を残しています。AIやゲーミング需要の恩恵をダイレクトに受けながら4%超のインカムを得られる、現在のSGXテックセクターの主役と言えます。
Frencken Group (E28)

分析: 半導体製造装置や医療機器の精密コンポーネントを手掛ける企業。52週間で +154.09%、直近13週間（約3ヶ月）だけでも +45.20% と猛烈に買われています。

注意点: 利回りは 0.94% と低く、期待先行でPEが 31.91倍 まで買われているため、配当目的の投資としては向きません。キャピタルゲイン（値上がり益）狙いの銘柄です。

## "Basic Materials": "materials",

In [62]:
sector_dfs['materials'].sort_values(by="rev", ascending=False, na_position="last").head(10).style.format({'SGX_URL': make_clickable})

,name,code,SGX_URL,price,roe,mkt_cap,rev,pe,yield,sector,gti,chg_4w,chg_13w,chg_26w,chg_52w,net_profit,p_cf,de,rev_chg_1y,pb
322,Le Tree Holdings,E6R,Link to SGX,0.004000,nan,34.586000,283025.000000,nan,nan,Basic Materials,79.300000,0.000000,0.000000,0.000000,300.00,0.180000,nan,nan,nan,64.100000
509,Sri Trang Agro,NC2,Link to SGX,0.720000,-2.55,1185.107000,105934.507000,nan,2.730000,Basic Materials,61.000000,5.110000,40.920000,52.540000,39.59,-0.010000,2.010000,0.440000,-0.78,0.580000
510,Sri Trang Gloves,STG,Link to SGX,0.355000,-0.29,1255.957000,22829.589000,nan,5.540000,Basic Materials,nan,0.000000,-1.360000,-3.890000,1.30,0.020000,8.020000,0.060000,-4.45,0.670000
133,ChinaSunsine,QES,Link to SGX,0.655000,9.37,624.466000,3277.392000,8.140000,3.050000,Basic Materials,69.400000,-1.870000,-1.160000,-12.560000,23.78,0.130000,5.250000,nan,-6.77,0.740000
230,Halcyon Agri,5VJ,Link to SGX,0.410000,-26.15,653.955000,2961.538000,nan,nan,Basic Materials,85.000000,0.000000,0.000000,0.000000,0.00,-0.030000,7.100000,1.310000,0.70,0.780000
302,Kep Infra Tr,A7RU,Link to SGX,0.510000,5.61,3104.037000,2277.479000,33.330000,7.730000,Basic Materials,90.600000,-4.670000,-1.920000,12.700000,40.87,0.060000,9.750000,1.550000,2.86,1.940000
590,Wilton Resources,5F7,Link to SGX,0.015000,-133.72,39.360000,2131.000000,nan,nan,Basic Materials,71.000000,25.000000,50.000000,25.000000,66.67,-176.370000,nan,0.020000,-66.39,13.160000
372,MSC,NPW,Link to SGX,0.715000,11.38,619.891000,1846.237000,15.920000,3.180000,Basic Materials,nan,17.210000,19.170000,45.920000,95.43,0.050000,nan,0.020000,3.97,2.420000
100,BRC Asia,BEC,Link to SGX,4.550000,19.05,1248.293000,1768.475000,11.980000,2.860000,Basic Materials,73.000000,3.410000,5.160000,16.960000,52.73,0.060000,5.320000,0.030000,4.84,2.350000
420,PanUnited,P52,Link to SGX,1.500000,18.25,1050.567000,898.436000,20.720000,3.000000,Basic Materials,75.500000,-1.320000,5.860000,53.500000,104.64,0.060000,13.140000,0.110000,10.60,3.620000


## "Consumer Defensive": "defensive",

In [63]:
sector_dfs['defensive'].sort_values(by="rev", ascending=False, na_position="last").head(10).style.format({'SGX_URL': make_clickable})

,name,code,SGX_URL,price,roe,mkt_cap,rev,pe,yield,sector,gti,chg_4w,chg_13w,chg_26w,chg_52w,net_profit,p_cf,de,rev_chg_1y,pb
269,Indofood Agri,5JS,Link to SGX,0.375000,8.50,523.464000,21056904.000000,5.790000,3.200000,Consumer Defensive,82.800000,-10.000000,3.200000,3.200000,22.86,0.070000,2.180000,0.030000,31.87,0.470000
104,Bumitama Agri,P8Z,Link to SGX,1.500000,19.17,2601.216000,19951443.000000,13.050000,6.230000,Consumer Defensive,78.400000,-27.540000,16.080000,17.970000,112.47,0.140000,7.830000,0.120000,19.24,2.430000
535,ThaiBev,Y92,Link to SGX,0.430000,17.46,10806.625000,328888.289000,11.100000,5.810000,Consumer Defensive,89.100000,2.560000,-0.940000,-2.160000,-2.16,0.080000,5.520000,1.340000,-2.06,1.860000
589,Wilmar Intl,F34,Link to SGX,3.430000,6.76,21413.143000,70415.698000,11.830000,4.080000,Consumer Defensive,84.400000,-9.020000,3.220000,15.360000,17.82,0.010000,7.070000,0.350000,4.51,0.760000
176,Emperador Inc.,EMI,Link to SGX,0.345000,3.74,4946.200000,56939.800000,66.690000,0.840000,Consumer Defensive,nan,0.000000,0.000000,-0.590000,7.05,0.070000,48.310000,0.390000,-7.47,2.420000
403,Olam Group,VC2,Link to SGX,1.240000,5.88,4676.225000,29599.871000,34.350000,4.030000,Consumer Defensive,81.800000,3.330000,46.750000,29.840000,36.96,0.000000,4.440000,1.060000,28.77,0.670000
216,Golden Agri-Res,E5H,Link to SGX,0.280000,7.61,3550.868000,12951.478000,6.910000,3.400000,Consumer Defensive,80.000000,-16.080000,-0.170000,3.400000,15.81,0.030000,2.360000,0.270000,18.72,0.510000
364,Mewah Intl,MV4,Link to SGX,0.295000,6.21,442.697000,5977.346000,6.440000,2.710000,Consumer Defensive,82.400000,-1.670000,-2.840000,2.100000,14.34,0.010000,nan,0.150000,25.00,0.380000
187,F & N,F99,Link to SGX,1.430000,5.01,2081.988000,2245.610000,14.870000,3.850000,Consumer Defensive,86.400000,-0.340000,0.350000,0.340000,20.73,0.060000,9.160000,0.280000,7.43,0.740000
131,ChinaKangdaFood,P74,Link to SGX,0.025000,-3.35,22.991000,1857.283000,nan,nan,Consumer Defensive,56.000000,-51.920000,-51.920000,-50.980000,0.00,-0.010000,3.060000,0.220000,12.53,0.120000


## "Real Estate": "estate",

In [64]:
sector_dfs['estate'].sort_values(by="rev", ascending=False, na_position="last").head(10).style.format({'SGX_URL': make_clickable})

,name,code,SGX_URL,price,roe,mkt_cap,rev,pe,yield,sector,gti,chg_4w,chg_13w,chg_26w,chg_52w,net_profit,p_cf,de,rev_chg_1y,pb
546,Tosei,S2D,Link to SGX,9.500000,15.27,1194.034000,109120.411000,24.520000,0.850000,Real Estate,nan,0.000000,0.000000,0.000000,4.36,0.160000,nan,1.510000,15.20,1.390000
598,Yanlord Land,Z25,Link to SGX,0.675000,0.84,1303.786000,14368.874000,25.680000,1.480000,Real Estate,73.800000,-4.860000,7.030000,-1.440000,39.80,0.030000,9.030000,0.420000,-60.52,0.210000
136,CityDev,C09,Link to SGX,8.360000,6.64,7468.838000,3587.092000,12.310000,2.990000,Real Estate,98.100000,2.200000,-7.820000,18.270000,67.44,0.020000,nan,1.130000,9.66,0.780000
202,Frasers Property,TQ5,Link to SGX,1.100000,2.33,4318.646000,3320.672000,25.000000,4.090000,Real Estate,94.900000,-3.510000,10.000000,4.090000,37.95,0.080000,4.370000,1.600000,-19.25,0.450000
572,UOL,U14,Link to SGX,10.010000,4.13,8474.418000,3234.085000,17.560000,1.800000,Real Estate,98.300000,-3.930000,-6.900000,18.890000,69.87,0.140000,6.550000,0.310000,15.72,0.720000
108,CapitaLandInvest,9CI,Link to SGX,2.520000,1.11,12583.699000,2133.000000,86.900000,4.760000,Real Estate,99.200000,-4.550000,-8.650000,1.150000,3.94,0.070000,25.910000,0.620000,-24.23,1.000000
227,GuocoLand,F17,Link to SGX,2.240000,2.03,2493.308000,1692.255000,23.260000,3.130000,Real Estate,86.000000,-9.680000,-11.110000,11.440000,57.14,0.060000,4.810000,1.020000,5.31,0.530000
113,CapLand IntCom T,C38U,Link to SGX,2.270000,5.89,17887.029000,1619.174000,18.020000,6.850000,Real Estate,106.500000,-4.220000,-3.360000,1.110000,16.62,0.590000,15.380000,0.570000,2.07,1.100000
109,CapLand Ascendas REIT,A17U,Link to SGX,2.470000,7.27,12328.661000,1538.574000,14.570000,7.590000,Real Estate,109.600000,-1.590000,-3.190000,-6.760000,0.49,0.580000,9.990000,0.570000,1.02,1.140000
432,PropNex,OYY,Link to SGX,1.790000,58.77,1324.600000,1116.416000,18.820000,5.310000,Real Estate,79.300000,0.560000,1.380000,-4.430000,81.25,0.060000,14.510000,0.020000,42.59,11.420000


## "Energy": "energy",

In [65]:
sector_dfs['energy'].sort_values(by="rev", ascending=False, na_position="last").head(10).style.format({'SGX_URL': make_clickable})

,name,code,SGX_URL,price,roe,mkt_cap,rev,pe,yield,sector,gti,chg_4w,chg_13w,chg_26w,chg_52w,net_profit,p_cf,de,rev_chg_1y,pb
124,China Aviation,G92,Link to SGX,1.840000,10.71,1582.738000,16439.557000,11.160000,2.700000,Energy,70.500000,-11.290000,0.510000,27.680000,124.95,0.010000,8.200000,0.000000,5.94,1.140000
466,Seatrium Ltd,5E2,Link to SGX,2.020000,4.89,6848.014000,11471.675000,21.330000,1.490000,Energy,nan,-15.130000,-11.640000,-3.300000,-1.91,0.020000,48.540000,0.420000,24.28,0.990000
497,Sinostar Pec,C9Q,Link to SGX,0.106000,1.76,101.760000,4380.806000,10.730000,nan,Energy,52.500000,-1.850000,11.580000,-7.020000,-35.37,0.010000,2.200000,0.070000,-17.15,0.320000
55,AnAn Intl,Y35,Link to SGX,0.019000,7.67,80.431000,2653.123000,7.920000,nan,Energy,54.200000,-5.000000,-13.640000,11.760000,375.00,0.000000,0.780000,0.130000,7.16,0.550000
212,Geo Energy Res,RE4,Link to SGX,0.450000,6.14,801.602000,492.055000,38.540000,0.890000,Energy,80.400000,-25.290000,0.440000,5.120000,22.70,0.050000,9.050000,0.560000,40.00,1.110000
358,Mermaid Maritime,DU4,Link to SGX,0.102000,3.44,192.856000,454.157000,9.350000,1.250000,Energy,53.700000,-14.290000,-19.310000,-16.030000,2.25,0.010000,3.060000,0.350000,-4.68,0.650000
448,Rex Intl,5WH,Link to SGX,0.072000,nan,97.736000,319.722000,nan,nan,Energy,81.300000,-21.740000,-59.090000,-52.320000,-47.83,-0.140000,0.560000,nan,6.97,nan
191,Federal Int,BDU,Link to SGX,0.245000,4.54,34.464000,128.334000,10.790000,2.040000,Energy,72.700000,6.250000,2.000000,28.140000,77.08,0.040000,2.200000,0.030000,189.95,0.470000
308,Kim Heng,5G2,Link to SGX,0.084000,-19.12,59.214000,120.989000,nan,nan,Energy,79.100000,-3.450000,-4.550000,-2.330000,9.09,-0.070000,3.280000,0.610000,-1.42,1.420000
93,Beng Kuang,BEZ,Link to SGX,0.495000,22.50,148.444000,98.155000,19.040000,1.210000,Energy,72.800000,-13.160000,23.750000,76.790000,181.25,0.050000,3.830000,0.240000,-12.27,5.620000


## "Utilities": "utilities",

In [66]:
sector_dfs['utilities'].sort_values(by="rev", ascending=False, na_position="last").head(10).style.format({'SGX_URL': make_clickable})

,name,code,SGX_URL,price,roe,mkt_cap,rev,pe,yield,sector,gti,chg_4w,chg_13w,chg_26w,chg_52w,net_profit,p_cf,de,rev_chg_1y,pb
483,SIIC Environment,BHK,Link to SGX,0.166000,5.64,427.561000,7072.781000,3.700000,6.630000,Utilities,56.100000,-3.280000,-1.120000,-1.670000,14.94,0.100000,1.330000,1.640000,-6.88,0.200000
468,Sembcorp Ind,U96,Link to SGX,6.130000,17.92,10912.566000,5799.000000,11.200000,4.080000,Utilities,94.300000,-5.980000,10.740000,5.010000,-4.06,0.140000,9.430000,1.550000,-9.63,1.970000
126,China Everbright,U9E,Link to SGX,0.225000,6.42,643.697000,5355.110000,4.670000,7.470000,Utilities,86.400000,-2.170000,0.830000,-7.240000,2.89,0.160000,3.500000,0.800000,-21.85,0.290000
610,Zheneng Jinjiang,BWM,Link to SGX,0.585000,9.24,838.960000,3784.870000,6.130000,6.320000,Utilities,68.900000,6.320000,5.420000,35.220000,36.70,0.200000,2.910000,0.950000,1.44,0.550000
522,Sunpower,5GD,Link to SGX,0.470000,14.37,386.064000,3336.654000,7.870000,nan,Utilities,54.400000,-6.930000,-3.090000,-14.550000,77.36,0.100000,3.420000,0.940000,-4.81,0.870000
145,CONCORD NE,SEG,Link to SGX,0.071000,1.62,587.872000,2307.541000,21.040000,nan,Utilities,nan,18.330000,42.000000,nan,nan,0.070000,1.280000,1.960000,-5.78,0.340000
412,Ouhua Energy,AJ2,Link to SGX,0.049000,-29.43,18.275000,2170.741000,nan,nan,Utilities,50.800000,58.060000,22.500000,6.520000,-18.33,-0.030000,nan,0.260000,-18.53,0.560000
208,Gallant Venture,5IG,Link to SGX,0.064000,1.64,349.642000,215.041000,34.410000,nan,Utilities,72.300000,-5.880000,4.920000,-24.710000,-28.89,0.030000,2.220000,0.010000,12.64,0.570000
128,China Intl,BEH,Link to SGX,0.035000,-1.41,2.744000,107.516000,nan,nan,Utilities,65.900000,6.060000,-22.220000,-25.530000,9.38,-0.030000,nan,0.220000,17.87,0.070000
446,Renaissance United,I11,Link to SGX,0.001000,-36.61,6.181000,79.816000,nan,nan,Utilities,61.600000,0.000000,0.000000,0.000000,0.00,-0.030000,3.460000,0.330000,-16.90,0.290000


Sembcorp Industries（U96）について、なぜこれほど強力な業績（ROE 17.92%）を持ちながらPE（PER）が11倍前後という「割安」な水準で放置されているのか、その理由を構造と背景から紐解きます。
1. なぜPEが11倍なのか？（4つの理由）
一般的にROEが15%を超えるような高収益企業は、期待値からPEが20〜30倍に買われることも珍しくありません。しかし、U96が11倍にとどまるのは以下のセクター固有の事情とマクロ環境が影響しています。

① 「資本集約型」の重厚長大ビジネスであるため
U96の主力は発電所やエネルギーインフラの建設・運営です。これらは巨額の設備投資（CapEx）を必要とし、借入金も多くなる（DE倍率も高めになる）ため、市場からはテクノロジー株のような「身軽な高成長株」としては評価されず、伝統的な「バリュー株・ユーティリティ株」の枠組み（PE10〜13倍が適正水準）でプライシングされます。

② シンガポール国内での電力卸売価格（スパークスプレッド）の頭打ち懸念
ここ数年、シンガポール国内の電力需給の逼迫から同社の電力事業は莫大な利益を上げてきましたが、市場は「この高い利益率は一時的なピークであり、今後は価格競争や供給増加によって緩やかに正常化（低下）していくのではないか」という警戒感を持っています。将来の利益減少を織り込むため、足元のPEは低く出やすくなります。

③ 「茶色（化石燃料）から緑（再生可能エネルギー）」への過渡期リスク
U96は現在、従来のガス発電（Brown）から、太陽光や風力などの再生可能エネルギー（Green）へ事業構造を大転換している真っ最中です。

2028年までに再エネ容量を25GWまで拡大する野心的な目標を掲げ、直近（2026年6月）にもオーストラリアのAlinta Energyの買収を完了させるなど猛スピードで動いています。

しかし、再エネは初期投資が重く、中国など一部地域での売電タラフ（価格制度）の不確実性もあるため、市場は「本当に計画通りの利益率を維持できるか」を慎重に見極めており、これが株価の重し（割安放置）に繋がっています。

④ 地理的リスク（新興国ビジネスの不確実性）
同社の再エネやインフラ事業はシンガポール国内だけでなく、中国、インド、ベトナム、インドネシアなど多岐にわたります。これらの新興国ビジネスは成長性が高い反面、規制変更や為替リスク、現地の電力網（グリッド）の制約といった不確実性を伴うため、シンガポール拠点の純粋なディフェンシブ株に比べてリスクプレミアム（ディスカウント）が要求されます。

## "Financial Services": "finance",

In [67]:
sector_dfs['finance'].sort_values(by="rev", ascending=False, na_position="last").head(10).style.format({'SGX_URL': make_clickable})

,name,code,SGX_URL,price,roe,mkt_cap,rev,pe,yield,sector,gti,chg_4w,chg_13w,chg_26w,chg_52w,net_profit,p_cf,de,rev_chg_1y,pb
390,Nomura Yen1k,N33,Link to SGX,1.990000,9.99,31638.361000,1854093.000000,nan,24.520000,Financial Services,nan,0.000000,0.000000,0.000000,10.86,0.210000,nan,4.020000,20.64,0.100000
158,DBS,D05,Link to SGX,63.780000,15.83,180984.341000,22938.000000,16.630000,4.890000,Financial Services,106.200000,10.320000,18.650000,20.750000,48.60,0.480000,16.420000,1.090000,3.05,2.610000
396,OCBC Bank,O39,Link to SGX,23.940000,12.05,107481.541000,14706.000000,14.500000,3.470000,Financial Services,103.200000,9.670000,17.660000,29.600000,53.60,0.510000,11.780000,0.490000,0.66,1.690000
222,Great Eastern,G07,Link to SGX,15.810000,12.66,14966.349000,14331.400000,12.550000,3.480000,Financial Services,92.000000,1.350000,0.940000,6.340000,-36.59,0.080000,nan,0.070000,13.75,1.440000
569,UOB,U11,Link to SGX,38.550000,9.09,63707.336000,13807.000000,14.020000,4.050000,Financial Services,104.300000,6.260000,9.030000,13.730000,14.37,0.340000,48.780000,0.880000,-3.47,1.240000
472,SGX,S68,Link to SGX,21.740000,31.15,23276.604000,1424.581000,35.870000,2.000000,Financial Services,99.800000,3.220000,22.970000,30.880000,56.94,0.450000,28.330000,0.300000,11.28,10.180000
570,UOB Kay Hian,U10,Link to SGX,3.750000,10.91,3690.706000,623.667000,14.960000,3.280000,Financial Services,66.500000,-6.670000,23.340000,56.170000,106.01,0.380000,nan,0.010000,17.08,1.630000
258,Hotung Inv,BLS,Link to SGX,1.590000,3.30,150.267000,478.860000,18.520000,6.480000,Financial Services,58.500000,5.160000,20.070000,20.070000,26.34,0.420000,7.370000,0.000000,15.29,0.620000
252,Hong Leong Fin,S41,Link to SGX,2.520000,2.97,1133.744000,183.445000,18.060000,3.530000,Financial Services,96.300000,-0.400000,2.440000,-0.330000,3.94,0.340000,nan,0.020000,-21.36,0.540000
486,Sing Inv & Fin,S35,Link to SGX,1.580000,8.85,373.573000,130.325000,8.830000,4.750000,Financial Services,97.800000,-3.660000,3.440000,6.770000,47.77,0.350000,nan,0.000000,-1.83,0.750000


## "Communication Services": "comm",

In [68]:
sector_dfs['comm'].sort_values(by="rev", ascending=False, na_position="last").head(10).style.format({'SGX_URL': make_clickable})

,name,code,SGX_URL,price,roe,mkt_cap,rev,pe,yield,sector,gti,chg_4w,chg_13w,chg_26w,chg_52w,net_profit,p_cf,de,rev_chg_1y,pb
493,Singtel,Z74,Link to SGX,4.290000,20.59,70384.353000,14260.600000,12.760000,3.050000,Communication Services,108.300000,-8.330000,-13.510000,-6.330000,15.56,0.220000,14.430000,0.370000,0.81,2.460000
494,Singtel 10,Z77,Link to SGX,4.280000,20.59,70384.353000,14260.600000,12.730000,nan,Communication Services,nan,-8.150000,-13.010000,-5.930000,10.88,0.220000,14.410000,0.370000,0.81,2.460000
516,StarHub,CC3,Link to SGX,1.010000,13.96,1743.250000,2352.800000,22.440000,5.940000,Communication Services,88.200000,-0.980000,2.970000,-8.770000,-5.31,0.040000,4.940000,2.910000,-0.63,3.480000
534,TeleChoice Intl,T41,Link to SGX,0.250000,17.27,113.594000,518.044000,17.860000,1.800000,Communication Services,77.300000,8.300000,45.430000,45.430000,162.37,0.010000,6.520000,0.030000,36.18,2.760000
213,GHY Culture,XJB,Link to SGX,0.091000,1.38,97.242000,511.244000,70.600000,1.100000,Communication Services,81.900000,-26.400000,-38.670000,-39.470000,-45.88,0.030000,nan,0.000000,22.42,0.980000
385,NetLink NBN Tr,CJLU,Link to SGX,0.985000,3.58,3838.517000,413.432000,46.030000,5.500000,Communication Services,106.500000,0.210000,3.280000,5.980000,20.14,0.200000,14.840000,0.450000,1.58,1.700000
68,Asian Pay Tv Tr,S7OU,Link to SGX,0.084000,2.23,151.734000,243.185000,9.030000,12.500000,Communication Services,60.600000,-2.330000,-2.990000,-12.500000,16.67,0.060000,1.280000,1.460000,-2.51,0.220000
480,Shopper360,1F0,Link to SGX,0.064000,-19.43,6.963000,192.655000,nan,nan,Communication Services,74.700000,-1.540000,-1.540000,28.000000,-8.57,-0.040000,nan,0.010000,1.97,0.330000
89,BACUI TECH,YYB,Link to SGX,0.002000,5.47,8.716000,60.345000,20.000000,0.550000,Communication Services,70.300000,100.000000,0.000000,0.000000,101.10,0.010000,nan,nan,7.56,1.070000
218,Goodwill,GEH,Link to SGX,0.145000,8.87,57.299000,50.790000,34.520000,5.170000,Communication Services,nan,0.000000,-2.680000,-10.490000,-10.49,0.030000,4.260000,0.880000,-4.15,3.180000
